# Chapter 2: The first-return identity

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## Statement

Let $v$ be drawn from the flux-weighted density $2v$ on $0<v<1$, start a projected contour on a plane
with outward speed $v$, and let $T$ be the reduced length at which it first returns to the plane. Then
$$F'(\lambda)=\tfrac12\Pr(T>\lambda),\qquad F(\lambda)=\tfrac12\,\mathbb E\min(T,\lambda).$$
**Status: exact.**

## Derivation 1: from the kinetic equation

Let $M_\lambda=\max_{\tau\le\lambda}Z_\tau$. By the symmetry $(Z,U)\to(-Z,-U)$ of the stationary
ensemble, $F=2\,\mathbb E M_\lambda$. For $y>0$ let $q(y,u;\lambda)=\Pr\{M_\lambda>y\mid U_0=u\}$. Because
$(Z,U)$ is Markovian, $q$ obeys the backward equation
$$\partial_\lambda q+u\,\partial_yq=\mathcal L_uq,\qquad \mathcal L_u=\tfrac12\partial_u(1-u^2)\partial_u,$$
with $q(0^+,u)=1$ for $u>0$ (a chain pointing at the plane reaches it at once) and $q\to0$ as $y\to\infty$.
Integrate over $y>0$ and $u\in[-1,1]$. The angular operator is conservative, $\int\mathcal L_uq\,du=0$.
The streaming term gives a boundary flux at $y=0$:
$\int_{-1}^1u\,q(0^+,u)\,du=\int_0^1u\,du-\int_0^1v\,q(0^+,-v)\,dv=\tfrac12-\int_0^1v\,q(0^+,-v)\,dv$.
Since $F=2\int\!\!\int q\,du\,dy\cdot\tfrac12$ (the factor $\tfrac12$ is the uniform angular average),
$$F'(\lambda)=\tfrac12-\int_0^1v\,q(0^+,-v;\lambda)\,dv .$$
For fixed $v>0$ the outgoing trace $q(0^+,-v;\lambda)$ is the probability that a contour started on the plane
with outward speed $v$ has returned by $\lambda$, i.e. the distribution function of $T_v$. Weighting with
$2v$ gives $\Pr(T>\lambda)=1-2\int_0^1v\,q(0^+,-v)\,dv=2F'(\lambda)$.

## Derivation 2: pathwise, without the kinetic equation

The span of a growing chain increases only while the endpoint is at its running maximum or minimum, at
rate $|U|$. Reversing the contour direction on $[0,\lambda]$ turns "the endpoint is a record" into "a
contour started on a plane, moving away from it, has not yet returned". Stationarity, time reversibility
and sign symmetry of $U$ are all that is used, together with boundedness of $U$; the initial speed is
weighted by $2v\varpi(v)/\mu$ with $\varpi$ the stationary density and $\mu=\mathbb E|U|$. For the WLC,
$\varpi=\tfrac12$ and $\mu=\tfrac12$. No independence of successive excursions is assumed.

## Numerical test

We estimate $\Pr(T>\lambda)$ by simulating flux-weighted returns with the heat-kernel sampler, and compare
with $2F'(\lambda)$ obtained by differentiating the practical formula of the article (which matches the
reference curve to 0.1 %).

In [2]:
from sphere_heat_kernel import heat_inverse, heat_step
import plot_main_figures as pmf
def survival(lams, h=0.01, N=20000, seed=7):
    """Flux-weighted first-return survival Pr(T > lambda) by direct simulation."""
    u, cdf, _ = heat_inverse(h)
    rng = np.random.default_rng(seed)
    v = np.sqrt(rng.random(N))                       # density 2v on (0,1)
    phi = 2*np.pi*rng.random(N)
    t = np.column_stack([np.sqrt(1-v*v)*np.cos(phi), np.sqrt(1-v*v)*np.sin(phi), v])  # outward speed v along e=z
    z = np.zeros(N); alive = np.ones(N, bool); out = {}
    lam_max = max(lams); steps = int(round(lam_max/h)); targets = {int(round(l/h)): l for l in lams}
    for k in range(1, steps+1):
        tn = heat_step(t, u, cdf, rng)
        z = z + 0.5*(t[:, 2] + tn[:, 2])*h; t = tn
        alive &= z > 0
        if k in targets: out[targets[k]] = alive.mean()
    return out
lams = [0.1, 0.3, 1.0, 3.0]
surv = survival(lams)
eps = 1e-4
for l in lams:
    dF = (pmf.rational(l*(1+eps)) - pmf.rational(l*(1-eps)))/(2*l*eps)
    print(f'lambda={l:4g}   2 F\'(lambda) = {2*dF:.4f}    simulated Pr(T>lambda) = {surv[l]:.4f} +/- {math.sqrt(surv[l]*(1-surv[l])/20000):.4f}')

lambda= 0.1   2 F'(lambda) = 0.9818    simulated Pr(T>lambda) = 0.9816 +/- 0.0010
lambda= 0.3   2 F'(lambda) = 0.9462    simulated Pr(T>lambda) = 0.9447 +/- 0.0016
lambda=   1   2 F'(lambda) = 0.8360    simulated Pr(T>lambda) = 0.8354 +/- 0.0026
lambda=   3   2 F'(lambda) = 0.6370    simulated Pr(T>lambda) = 0.6372 +/- 0.0034


The simulated survival probabilities agree with $2F'$ within sampling error (the practical formula
contributes an error of order $10^{-3}$ to $F'$ near $\lambda\sim1$, where its curvature is largest).

## Consequences checked on the reference table

* $\Pr(T>\lambda)$ cannot increase, so $F'$ is nonincreasing: $F$ is **concave**, and $F(\lambda)/\lambda$ is nonincreasing.
* $0<F'\le\tfrac12$ gives $F\le\lambda/2$ (the pathwise bound $X_c\le L$ alone would only give $F\le\lambda$).
* Subadditivity $F(\lambda_1+\lambda_2)\le F(\lambda_1)+F(\lambda_2)$ follows from concatenating contours.

In [3]:
d1 = np.diff(F)/np.diff(LAM)                      # chord slopes between consecutive nodes
print('all chord slopes positive        :', bool(np.all(d1 > 0)))
print('chord slopes nonincreasing       :', bool(np.all(np.diff(d1) <= 1e-12)), '  (concavity)')
print('F/lambda nonincreasing           :', bool(np.all(np.diff(F/LAM) <= 0)))
print('F <= lambda/2 at every node      :', bool(np.all(F <= LAM/2)))
print('max chord slope                  :', d1.max().round(6), ' (<= 1/2)')
# subadditivity on pairs of nodes whose sum is also a node
pairs = [(a, b) for a in LAM for b in LAM if a <= b and any(np.isclose(a+b, LAM))]
Fi = lambda x: F[np.argmin(np.abs(LAM-x))]
print('subadditivity on', len(pairs), 'node pairs:', all(Fi(a+b) <= Fi(a)+Fi(b)+1e-12 for a, b in pairs))

all chord slopes positive        : True
chord slopes nonincreasing       : True   (concavity)
F/lambda nonincreasing           : True
F <= lambda/2 at every node      : True
max chord slope                  : 0.498919  (<= 1/2)
subadditivity on 315 node pairs: True


## The pathwise derivation, one step at a time

The article's derivation has five steps. Each is stated here and then tested on sampled chains.

**Step 1: the maximum advances only at a record.** $M_\tau=\max_{\tau'\le\tau}Z_{\tau'}$ is nondecreasing,
and it changes only while $Z_\tau=M_\tau$ and the contour moves outward. Since $dZ=U\,d\tau$,
$$\frac{dM_\tau}{d\tau}=U_\tau^+\,\mathbf 1_{\{Z_\tau=M_\tau\}},\qquad U^+=\max(U,0),$$
and likewise $-dm_\tau/d\tau=U_\tau^-\mathbf 1_{\{Z_\tau=m_\tau\}}$ for the minimum.

**Step 2: average (the record form of the growth rate).** $F=\langle M_\lambda-m_\lambda\rangle$, and the symmetry $(Z,U)\to(-Z,-U)$ makes
the two extremes contribute equally: $F'(\lambda)=2\langle U_\lambda^+\mathbf 1_{\{Z_\lambda=M_\lambda\}}\rangle$.
Only the *end* of the contour enters.

**Step 3: read the contour backward.** Let $Y_s=Z_\lambda-Z_{\lambda-s}$, $0\le s\le\lambda$, be the height of
the end above the contour point a reduced length $s$ before it. The end is the maximum if and only if
$Y_s\ge0$ for all $s$. Now $dY_s/ds=U_{\lambda-s}$, the reversed tangent. Rotational diffusion in
equilibrium is time reversible, so the reversed tangent is again a stationary rotational diffusion.
$Y$ is therefore the position of a contour that starts on a plane ($Y_0=0$) with speed $U_\lambda=v$
away from it, and the event is "no return to the plane within $\lambda$".

**Step 4: weight the speeds (the first-return identity).** $U_\lambda$ is uniform on $[-1,1]$ (density $\tfrac12$), so
$$F'(\lambda)=2\int_0^1\tfrac12\,v\,\Pr\nolimits_v(T>\lambda)\,dv=\tfrac12\int_0^1 2v\,\Pr\nolimits_v(T>\lambda)\,dv
=\tfrac12\Pr(T>\lambda),$$
with $v$ drawn from the density $2v$. Integrating from $F(0)=0$:
$F(\lambda)=\tfrac12\int_0^\lambda\Pr(T>\ell)\,d\ell=\tfrac12\,\mathbb E\min(T,\lambda)$.

**Step 5: the null touch.** The reversed event in step 3 has a weak inequality ($Y_s\ge0$), whereas $T$
is the first *strictly positive* return. They differ only if the contour touches the plane without
crossing it, which requires $(Z,U)$ to pass through the single point $(0,0)$. For this process that has
probability zero (supplementary material).

**Consequence: the derivative with respect to $p$ .** With $X=pF(L/p)$ at fixed $L$,
$\partial X/\partial p=F-\lambda F'$. From step 4,
$F-\lambda F'=\tfrac12\mathbb E[\min(T,\lambda)-\lambda\mathbf 1_{\{T>\lambda\}}]=\tfrac12\mathbb E[T\,\mathbf 1_{\{T\le\lambda\}}]\ge0$.

### Test of steps 1-2: growth of the span at the end of equilibrium chains

We sample equilibrium chains, look only at their ends, and average $|U_\lambda|$ over the chains whose end
is an outward record. This estimates $F'(\lambda)$ directly from the record form of the growth rate, without any reversal. It is
compared with the slope of the spectral solution, obtained by a central difference of the solver of
Chapter 6 at one angular order.

In [4]:
from wlc_pn_transfer import invert_alias_corrected, BromwichSettings
_st = BromwichSettings(12.0, 32, 32)
def F_and_slope(l, rel=0.005, m=64):
    """F and F' from the spectral solver at one angular order (central difference; the finite-order error largely cancels in the slope)."""
    lo_, mid, hi_ = (invert_alias_corrected(x, m, _st) for x in (l*(1-rel), l, l*(1+rel)))
    return mid, (hi_ - lo_)/(2*rel*l)
SPEC = {l: F_and_slope(l) for l in (0.1, 0.3, 1.0, 3.0)}
Fval = lambda l: SPEC[l][0]; Fprime = lambda l: SPEC[l][1]

def end_record_rate(lam, h=0.005, N=40000, seed=11):
    """Mean growth rate of the span at the chain end: <|U| 1{end is an outward record}> (the record form of the growth rate)."""
    u, cdf, _ = heat_inverse(h)
    rng = np.random.default_rng(seed)
    t = rng.normal(size=(N, 3)); t /= np.linalg.norm(t, axis=1)[:, None]
    pos = np.zeros((N, 3)); mx = pos.copy(); mn = pos.copy()
    for _ in range(int(round(lam/h))):
        tn = heat_step(t, u, cdf, rng)
        turn = t*tn < 0                                      # the projected tangent changes sign inside the step
        ext = pos + 0.5*h*t*t/np.where(turn, t - tn, 1.0)    # turning point of the quadratic position interpolant
        mx = np.where(turn, np.maximum(mx, ext), mx); mn = np.where(turn, np.minimum(mn, ext), mn)
        pos = pos + 0.5*(t + tn)*h; t = tn
        mx = np.maximum(mx, pos); mn = np.minimum(mn, pos)
    at_max = (pos >= mx) & (t > 0)                   # end is the maximum and points outward
    at_min = (pos <= mn) & (t < 0)                   # end is the minimum and points outward
    growth = (np.abs(t)*(at_max | at_min)).mean(axis=1)          # the three axes averaged within each chain
    return growth.mean(), growth.std()/math.sqrt(N), (at_max | at_min).mean()
for l in (0.3, 1.0, 3.0):
    g, se, frac = end_record_rate(l)
    print(f'lambda={l:3g}   <|U| 1(record)> = {g:.4f} +/- {se:.4f}    F\'(lambda), spectral = {Fprime(l):.4f}    (end is a record in {100*frac:.0f} % of chain-axis pairs)')

lambda=0.3   <|U| 1(record)> = 0.4723 +/- 0.0004    F'(lambda), spectral = 0.4725    (end is a record in 84 % of chain-axis pairs)


lambda=  1   <|U| 1(record)> = 0.4191 +/- 0.0005    F'(lambda), spectral = 0.4185    (end is a record in 71 % of chain-axis pairs)


lambda=  3   <|U| 1(record)> = 0.3192 +/- 0.0007    F'(lambda), spectral = 0.3187    (end is a record in 53 % of chain-axis pairs)


### Test of steps 3-4: return lengths of flux-weighted starts

Now the other side of the identity. Contours are started *on* a plane with outward speed drawn from the
density $2v$, and the return length $T$ of each is recorded (up to $\lambda=3$). The two forms of the first-return identity and
the expression for $\partial X/\partial p$ are then averages over the same sample of $T$.

In [5]:
def return_lengths(lam_max, h=0.005, N=40000, seed=7):
    """First-return lengths T of contours started on the plane with outward speed of density 2v."""
    u, cdf, _ = heat_inverse(h)
    rng = np.random.default_rng(seed)
    v = np.sqrt(rng.random(N)); phi = 2*np.pi*rng.random(N)
    t = np.column_stack([np.sqrt(1-v*v)*np.cos(phi), np.sqrt(1-v*v)*np.sin(phi), v])
    z = np.zeros(N); T = np.full(N, np.inf)
    for k in range(1, int(round(lam_max/h)) + 1):
        tn = heat_step(t, u, cdf, rng)
        a, b = t[:, 2], tn[:, 2]
        zn = z + 0.5*(a + b)*h
        dip = (a < 0) & (b > 0)                              # interior minimum of the quadratic position interpolant
        zmin = np.where(dip, z + 0.5*h*a*a/np.where(dip, a - b, 1.0), zn)
        hit = np.isinf(T) & (np.minimum(zn, zmin) <= 0)
        c2 = (b[hit] - a[hit])/(2*h)                         # first root of z + a x + c2 x^2 = 0 within the step
        disc = np.maximum(a[hit]**2 - 4*c2*z[hit], 0.0); quad = np.abs(c2) > 1e-14
        x = np.where(quad, (-a[hit] - np.sqrt(disc))/(2*np.where(quad, c2, 1.0)), -z[hit]/np.where(a[hit] != 0, a[hit], -1.0))
        T[hit] = (k-1)*h + np.clip(x, 0, h)
        z, t = zn, tn
    return T
T = return_lengths(3.0); n = len(T)
print('lambda   (1/2)Pr(T>lambda)     F\'(spectral) | (1/2)E min(T,lambda)       F(spectral) | (1/2)E[T 1(T<=lambda)]     F - lambda F\'')
for l in (0.1, 0.3, 1.0, 3.0):
    surv = (T > l).mean(); m = np.minimum(T, l); w = np.where(T <= l, T, 0.0)
    print(f'{l:5g}   {surv/2:.4f} +/- {math.sqrt(surv*(1-surv)/n)/2:.4f}   {Fprime(l):.4f}       | '
          f'{m.mean()/2:.5f} +/- {m.std()/2/math.sqrt(n):.5f}   {Fval(l):.5f}     | '
          f'{w.mean()/2:.5f} +/- {w.std()/2/math.sqrt(n):.5f}      {Fval(l)-l*Fprime(l):.5f}')

lambda   (1/2)Pr(T>lambda)     F'(spectral) | (1/2)E min(T,lambda)       F(spectral) | (1/2)E[T 1(T<=lambda)]     F - lambda F'
  0.1   0.4907 +/- 0.0003   0.4905       | 0.04955 +/- 0.00002   0.04952     | 0.00048 +/- 0.00002      0.00047
  0.3   0.4721 +/- 0.0006   0.4725       | 0.14580 +/- 0.00010   0.14581     | 0.00417 +/- 0.00010      0.00405
    1   0.4171 +/- 0.0009   0.4185       | 0.45629 +/- 0.00057   0.45691     | 0.03922 +/- 0.00053      0.03844
    3   0.3171 +/- 0.0012   0.3187       | 1.17961 +/- 0.00247   1.18328     | 0.22839 +/- 0.00197      0.22709


The equilibrium-chain average of step 2 and the flux-weighted return statistics of step 4 give the same
$F'$, and both agree with the spectral solution of the kinetic equation, a third route. All uncertainties are
one sampling standard error, and the differences are within about one and a half of them. The paths have
a step of $0.005$; records and returns inside a step are located with the quadratic position interpolant,
because counting them only at the nodes biases both estimates by more than the sampling error.

## Transform and derivatives (additional to the article)

For real $s>0$, $\widehat F(s)=\int_0^\infty e^{-s\lambda}F\,d\lambda=(1-\mathbb Ee^{-sT})/(2s^2)$. At fixed $L$,
$\partial_pX=F-\lambda F'=\tfrac12\mathbb E[T\,\mathbf 1_{T\le\lambda}]\ge0$ and the Hessian of $X$ in $(L,p)$ is
$p^{-1}F''\begin{pmatrix}1&-\lambda\\-\lambda&\lambda^2\end{pmatrix}$, negative semidefinite because
$F''\le0$: $X$ is jointly concave in its two length scales.

In [6]:
lam, p, L = sp.symbols('lambda p L', positive=True)
Fs = sp.Function('F')
X = p*Fs(L/p)
H = sp.hessian(X, (L, p)).applyfunc(sp.simplify)
H = H.subs(L, lam*p).applyfunc(sp.simplify)
display(H)
print('determinant:', sp.simplify(H.det()), '   trace sign = sign of F\'\'(lambda) * (1+lambda^2)/p')

Matrix([
[        Derivative(F(lambda), (lambda, 2))/p,   -lambda*Derivative(F(lambda), (lambda, 2))/p],
[-lambda*Derivative(F(lambda), (lambda, 2))/p, lambda**2*Derivative(F(lambda), (lambda, 2))/p]])

determinant: 0    trace sign = sign of F''(lambda) * (1+lambda^2)/p
